# Mesh-fidelity evaluation: naive CloudCompare vs EPFL clean

Quantifies **how faithfully a sampled colored cloud reproduces its source mesh**, comparing
**A = naive CloudCompare sampling** vs **B = EPFL de-speckled sampling** on the same models.

Five metrics (all **lower = better**), computed by `scripts/eval_fidelity.py`:

| metric | meaning | uncolored ancestor |
|---|---|---|
| `speckle` | local color variance among kNN (color noise) | our double-face metric |
| `uniformity` | CV of nearest-neighbor distance (even spread) | PU-Net NUC |
| `p2f` | point-to-surface distance (points on surface) | Metro / PU-GAN |
| `coverage` | surface→cloud distance / bbox (surface covered) | PU-GAN bidirectional |
| `color_dE` | CIELAB ΔE vs mesh texture (color correctness) | CIELAB + texel lookup |

Set `N_EVAL_PER_CAT` small for a quick demo; raise to cover the whole set.

In [ ]:
# ============================================================
# Cell A - Configuration
# ============================================================
import os

ORTAK = os.environ.get("PCC_DATA_ROOT", os.path.abspath("data"))  # set PCC_DATA_ROOT to your data folder
SHARED_DATA_DIR = f"{ORTAK}/shapenetcore"          # the ShapeNet zips
OUT_DIR = f"{ORTAK}/eval"                           # results (CSV) land here
WORK_DIR = os.path.abspath("work_eval")

CATEGORIES = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
SYNSETS = list(CATEGORIES.values())
NAME_BY_SYNSET = {v: k for k, v in CATEGORIES.items()}

N_EVAL_PER_CAT = 2         # models per category to evaluate (raise for the full run)
N_POINTS       = 1_048_576 # sampling density (2^20), matches sample_s2
SEED           = 42        # same selection as sample_s2 -> evaluate the dataset's own models

In [ ]:
# ============================================================
# Cell B - Deps, imports, CloudCompare + eval module
# ============================================================
import shutil, subprocess, sys, glob

if shutil.which("apt-get"):     # Colab/Ubuntu; on macOS: brew install --cask cloudcompare
    subprocess.run("apt-get update -qq && apt-get install -y -qq cloudcompare xvfb", shell=True, check=True)
subprocess.run("pip install -q -U open3d pymeshlab trimesh rtree pillow", shell=True, check=True)

# import the eval module from scripts/ (repo-root or notebooks/ cwd both handled)
for c in ("scripts", "../scripts"):
    p = os.path.abspath(c)
    if os.path.isdir(p):
        sys.path.insert(0, p); break
import eval_fidelity as ev

CC_BIN = (os.environ.get("PCC_CC_BIN")
          or shutil.which("CloudCompare") or shutil.which("cloudcompare")
          or next((p for p in ["/Applications/CloudCompare.app/Contents/MacOS/CloudCompare"]
                   if os.path.exists(p)), None))
assert CC_BIN, "CloudCompare not found (brew install --cask cloudcompare, or set PCC_CC_BIN)."
os.makedirs(WORK_DIR, exist_ok=True)
print("CloudCompare:", CC_BIN)

In [ ]:
# ============================================================
# Cell C - Helpers: extract one model, sample with CloudCompare
# ============================================================
import zipfile, random
import numpy as np

def cc_sample(obj_path, out_ply, n=N_POINTS):
    xvfb = "xvfb-run -a " if shutil.which("xvfb-run") else ""
    cmd = (f'{xvfb}"{CC_BIN}" -SILENT -AUTO_SAVE OFF -C_EXPORT_FMT PLY '
           f'-O "{obj_path}" -SAMPLE_MESH POINTS {n} -SAVE_CLOUDS')
    subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
    prod = [p for p in glob.glob(os.path.join(os.path.dirname(obj_path), "*.ply"))
            if "SAMPLED_POINTS" in os.path.basename(p)]
    if not prod:
        raise RuntimeError("no PLY from " + obj_path)
    os.makedirs(os.path.dirname(out_ply), exist_ok=True); shutil.move(prod[0], out_ply)
    return out_ply

def extract_model(zip_path, mid, dest):
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        d = next(n for n in names if n.endswith("models/model_normalized.obj") and mid in n)
        model_dir = d.split("models/")[0]
        zf.extractall(dest, members=[n for n in names if n.startswith(model_dir)])
    return os.path.join(dest, model_dir, "models", "model_normalized.obj")

def select_models(synset):
    with zipfile.ZipFile(f"{SHARED_DATA_DIR}/{synset}.zip") as zf:
        ids = sorted({n[:n.index("models/")].rstrip("/").split("/")[-1]
                      for n in zf.namelist() if n.endswith("models/model_normalized.obj")})
    return random.Random(SEED).sample(ids, min(N_EVAL_PER_CAT, len(ids)))

In [ ]:
# ============================================================
# Cell D - Run A-vs-B on the selected models
# ============================================================
import traceback
import pandas as pd
from tqdm import tqdm

jobs = [(s, mid) for s in SYNSETS for mid in select_models(s)]
rows, errors = [], []

for synset, mid in tqdm(jobs, desc="A-vs-B fidelity"):
    md_dir = os.path.join(WORK_DIR, mid); os.makedirs(md_dir, exist_ok=True)
    try:
        obj = extract_model(f"{SHARED_DATA_DIR}/{synset}.zip", mid, md_dir)
        clean = ev.build_clean_reference(obj, os.path.join(md_dir, "clean"))   # exterior reference
        ply_A = cc_sample(obj,   os.path.join(md_dir, "A.ply"))                 # naive (speckle)
        ply_B = cc_sample(clean, os.path.join(md_dir, "B.ply"))                 # EPFL clean
        for method, ply in [("A_random", ply_A), ("B_epfl", ply_B)]:
            m = ev.evaluate(ply, clean)
            m.update(method=method, category=NAME_BY_SYNSET[synset], model_id=mid)
            rows.append(m)
    except Exception:
        errors.append((synset, mid, traceback.format_exc()))
    finally:
        shutil.rmtree(md_dir, ignore_errors=True)   # keep disk small

df = pd.DataFrame(rows)
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(f"{OUT_DIR}/fidelity.csv", index=False)
print(f"evaluated {len(jobs)} models, {len(errors)} failed -> {OUT_DIR}/fidelity.csv")

In [ ]:
# ============================================================
# Cell E - Aggregate A vs B (mean over models)
# ============================================================
METRICS = ["speckle", "uniformity", "p2f", "coverage", "color_dE"]
if len(df):
    overall = df.groupby("method")[METRICS].mean()
    print("=== overall (mean over all models) ===")
    print(overall.to_string(float_format=lambda x: f"{x:.4g}"))
    print("\n=== per category ===")
    print(df.groupby(["category", "method"])[METRICS].mean().to_string(float_format=lambda x: f"{x:.4g}"))